# Predict


In [0]:
!pip install feature-engine

In [0]:
import mlflow

# IMPORTANDO O MODELO
model = mlflow.sklearn.load_model("models:/workspace.analytics_rm.tmw_ativacao_t5/1")
model

In [0]:
# IMPORTANDO DADOS NOVOS
df = spark.sql(""" 
SELECT
    * EXCEPT (t2.dtRef, t2.idCliente)
FROM workspace.analytics_rm.fs_loyalty AS t1
LEFT JOIN workspace.analytics_rm.fs_education AS t2
    ON t1.IdCliente = t2.idCliente AND t1.dtRef = t2.dtRef
WHERE t1.dtRef = (SELECT MAX(dtRef) FROM workspace.analytics_rm.fs_loyalty)
""").toPandas()

df.head(5)

In [0]:
#
import os
notebook_path = os.getcwd()

notebook_path

In [0]:
# 
X = df[model.feature_names_in_]
predict = model.predict_proba(X)[:, 0]
df['Proba'] = predict
top_50_df = df.sort_values(by='Proba', ascending=True).head(50)

top_50_df.to_csv(os.path.join(notebook_path,'output/top_50_predictions.csv'), index=False)


# Feature Importance




In [0]:
import pandas as pd
import numpy as np

# 1. Tenta extrair a assinatura (todas as colunas) direto dos metadados do MLflow
try:
    # Busca os metadados do modelo registrado para pegar os nomes das colunas
    model_meta = mlflow.models.get_model_info("models:/workspace.analytics_rm.tmw_ativacao_t5/1")
    feature_names = model_meta.signature.inputs.input_names()
    print(f"✅ {len(feature_names)} colunas detectadas automaticamente.")
except Exception:
    print("⚠️ Não foi possível ler a assinatura do MLflow.")
    print("Por favor, substitua a linha abaixo pelo seu DataFrame de treino:")
    # Se der erro, use o seu dataframe de treino: feature_names = list(X_train.columns)
    feature_names = None 

# 2. Extrai a importância das variáveis do objeto 'model' que você acabou de carregar
if feature_names:
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        df_importance = pd.DataFrame({"Feature": feature_names, "Importance": importances})
        df_importance = df_importance.sort_values(by="Importance", ascending=False)
        display(df_importance)
        
    elif hasattr(model, "coef_"):
        coefs = np.abs(model.coef_)
        if coefs.ndim > 1: coefs = np.mean(coefs, axis=0) # Média se for multiclasse
        df_importance = pd.DataFrame({"Feature": feature_names, "Importance": coefs})
        df_importance = df_importance.sort_values(by="Importance", ascending=False)
        display(df_importance)
        
    else:
        print("Este algoritmo não possui os atributos nativos '.feature_importances_' ou '.coef_'.")


In [0]:
import pandas as pd

# 1. Armazena as importâncias que descobrimos no RandomForest
importances = model.steps[-1][1].feature_importances_

# 2. BUSCA AUTOMÁTICA DOS NOMES (Substitua 'X_train' pelo nome do seu DataFrame de treino)
# Se o seu DataFrame for do Spark (PySpark), use: feature_names = X_train.columns
# Se for do Pandas, use: feature_names = list(X_train.columns)
try:
    # Digite o nome da sua variável de treino aqui se não for X_train
    feature_names = list(X.columns) 
except NameError:
    print("⚠️ A variável 'X_train' não foi encontrada neste notebook.")
    print("Preencha a lista abaixo manualmente com os nomes das suas 8 colunas na ordem do treino:")
    feature_names = ["coluna1", "coluna2", "coluna3", "coluna4", "coluna5", "coluna6", "coluna7", "coluna8"]

# 3. Valida e exibe o resultado final
if len(feature_names) == len(importances):
    df_importance = pd.DataFrame({"Feature": feature_names, "Importance": importances})
    df_importance = df_importance.sort_values(by="Importance", ascending=False)
    
    # Exibe a tabela interativa com os nomes reais das 8 variáveis
    display(df_importance)
else:
    print(f"❌ Erro de mapeamento: Sua lista/DataFrame tem {len(feature_names)} colunas, "
          f"mas o modelo espera exatamente {len(importances)} features.")


In [0]:
df_importance.to_csv(os.path.join(notebook_path,'output/df_importances.csv'), index=False)

In [0]:
# Converta o DataFrame do Pandas para Spark DataFrame
spark_df = spark.createDataFrame(df[['IdCliente', 'Proba']])

# Salve como uma tabela Delta permanente
spark_df.write.format("delta").mode("overwrite").saveAsTable("workspace.model_output.tmw_ativacao_t5_scores")
